# Module 5: Why Search Feels Smart

This week we de-magic the line **"search feels smart."** By the end you'll have built the
**retrieval engine** of a RAG bot:

1. An **embedding** turns text into a list of numbers (a *vector*).
2. **Cosine similarity** measures how much two vectors *point the same way* = how similar two
   pieces of text are. We build it **by hand in pure Python** first.
3. The **nearest-chunk loop** scores every candidate chunk and returns the best one. It's the
   `for`-loop you already know — it just scores options instead of printing them.
4. A **data-boundary check** distinguishes local chat from fully local RAG.

Then we peek at the **NumPy** version (`np.dot` / `np.linalg.norm`) — the *real* path once
vectors get big. Same math, far faster.


## Warm-up 
**Predict both answers, then run.**

1. `history` is a list of dicts. What does `history[0]["role"]` give you?
2. `safe_clean(None)` returns `""` instead of crashing. Which two words made that possible?

Chatbots should be able to retrieve information as well

In [1]:
my_answer_1 = ""   # <- what does history[0]["role"] give you? (a word)
my_answer_2 = ""   # <- the two keywords that stop a crash (e.g. "aaa/bbb")

# Week 3: a list of dicts, read list-position first, then dict-key.
history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "hi"},
]
print("history[0]['role'] ->", history[0]["role"])

# Week 4: the same crash-proofing you wrote as safe_clean.
def safe_first_word(text):
    try:
        return text.split()[0]
    except (AttributeError, IndexError):
        return ""

print("safe_first_word(None) ->", repr(safe_first_word(None)))

# Self-check (advises, never breaks)
hits = 0
if my_answer_1.strip().lower() == "system":
    hits += 1
else:
    print("HINT  1: messages[0] is the SYSTEM message — position 0, then the 'role' key.")
if "try" in my_answer_2.lower() and "except" in my_answer_2.lower():
    hits += 1
else:
    print("HINT  2: try/except — the pattern you'll wrap around this week's embedding call.")
print(f"\n{hits}/2 recalled. Both of these come back today: vectors are LISTS you index,")
print("and a bad vector is exactly what try/except protects your retriever from.")

history[0]['role'] -> system
safe_first_word(None) -> ''
HINT  1: messages[0] is the SYSTEM message — position 0, then the 'role' key.
HINT  2: try/except — the pattern you'll wrap around this week's embedding call.

0/2 recalled. Both of these come back today: vectors are LISTS you index,
and a bad vector is exactly what try/except protects your retriever from.


## 0. Make the course package importable

The code you build all term lives in the `rag-starter/` repo. Run this once in a terminal from
inside that folder, then select **`rag-starter/.venv`** as the notebook kernel:

```bash
uv venv && uv pip install -e '.[dev]'
```

Because this is an editable install, changes you make to `ribot` are available immediately.
If the package is unavailable, the next cell still uses local offline fallback definitions.

In [ ]:
# GIVEN — run as-is. Confirm that this notebook kernel can import the course package.
try:
    import ribot
    print("✓ Connected to the editable ribot package.")
except ModuleNotFoundError:
    print("ribot is not installed in this kernel — using local fallback definitions below.")

In [5]:
# GIVEN — run as-is, don't study it. It hands you an `embed()` function; how the
# offline stand-in works inside is not this week's material (see Modules/IDIOMS.md).
import math

try:
    from ribot.embeddings import fallback_embed, get_embedder
    print("Imported the real embedders from ribot.embeddings — same code you grow all term.")
except ModuleNotFoundError:
    # Local fallback: a tiny stand-in so the lesson always runs offline.
    import hashlib, re
    EMBED_DIM = 256
    _STOP = set("a an the of to in on at for and or but if is are how do i my you your it this".split())
    def _tokens(text):
        return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in _STOP]
    def fallback_embed(text):
        vec = [0.0] * EMBED_DIM
        for tok in _tokens(text):
            vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % EMBED_DIM] += 1.0
        norm = math.sqrt(sum(v * v for v in vec))
        return [v / norm for v in vec] if norm else vec
    def get_embedder():
        return fallback_embed
    print("Using local fallback definitions (same idea as ribot.embeddings).")

embed = get_embedder()  # offline: this is fallback_embed; with a key: the real API

Imported the real embedders from ribot.embeddings — same code you grow all term.


## 1. What is an embedding? Text → a list of numbers

An **embedding** is a function that takes a piece of text and returns a fixed-length list of
numbers — a **vector**. The magic property of a *real* embedding model: texts with **similar
meaning** come out as vectors that *point the same way*, even when they share no words.

Real embedding models return ~1536 numbers per text. Our offline `fallback_embed()` returns
256 — but be clear about what it is: a **hashed word-counter**. It puts texts close together
only when they **share words**. 
Later in this notebook we'll run an honest test that shows exactly where it falls short (and what a
real model buys you). Let's look at one vector first.

### 1a. Where do the numbers come from?

Week 2 showed: your text is chopped into **tokens**, and each token
becomes a list of numbers. How?

**Step one is the same tokenizer from Week 2.** `"how do I reset my password?"` is split into
word-ish pieces before anything numeric happens. Each piece has a fixed starting vector, learned
during pretraining: a first guess at what that piece means, identical every time that token appears.

**Step two is the part that makes it useful — and it is what "attention" does.** Those starting
vectors get adjusted by *the other tokens in the sentence*. Each token looks at its neighbours and
its own vector shifts to account for them. Run this pair through a real embedding model:

```
"I sat on the river bank"          →  "bank" ends up near  water, shore, mud
"I deposited it at the bank"       →  "bank" ends up near  money, branch, teller
```

Same token, same starting vector, two very different places to land — because in one sentence
`river` was next to it and in the other `deposited` was. That adjustment, stacked many times over,
is a **transformer**. The word describes the architecture doing the adjusting; there is nothing else
hiding behind the term.

That is also the honest reason the offline `fallback_embed()` below cannot do this. It is a hashed
word-counter: it sees which words are present and nothing about what surrounds them, so both
sentences above put `bank` in exactly the same spot. Watch for that when you run the
meaning-vs-words test in §2 — the failure you will see is precisely this missing step.

> **What stays out of scope, honestly:** how those vectors get *learned* — the optimization that
> turns a randomly-initialized network into one where `river bank` and `money bank` separate. That
> is the ML foundations track (see the Week 10 map), and you do not need it to build, measure, or
> debug retrieval. What you need is the sentence above: **similar meaning, similar direction** — and
> from here on, that direction is all we measure.

In [6]:
vec = embed("how do I reset my password?")
print("type:", type(vec).__name__)
print("length (dimensions):", len(vec))
print("first 5 numbers:", [round(x, 3) for x in vec[:5]])

# A vector is just a list of floats. Nothing magic — you can index it like any list.
print("vec[0]:", round(vec[0], 3))

type: list
length (dimensions): 256
first 5 numbers: [0.0, 0.0, 0.0, 0.0, 0.0]
vec[0]: 0.0


## 2. Cosine similarity — the math, by hand

We want ONE number that says *how similar* two vectors are. The standard answer is **cosine
similarity**: the cosine of the angle between them.

$$\cos(a, b) = \frac{a \cdot b}{\lVert a\rVert \; \lVert b\rVert}$$

Three pieces, all things you can compute with a `for`-loop:

- **Dot product** `a · b` — multiply matching elements, add them up: `sum(a[i]*b[i])`.
- **Magnitude** `‖a‖` — the vector's length: `sqrt(sum(a[i]**2))`.
- **Divide** the dot product by the two magnitudes.

Result ranges from **1.0** (point the exact same way = same meaning) down toward **0.0**
(unrelated). Let's build each piece in pure Python.

In [7]:
def dot(a, b):
    """Dot product: multiply matching elements and add them up."""
    return sum(x * y for x, y in zip(a, b))

def magnitude(a):
    """Length of a vector: sqrt of the sum of squares."""
    return math.sqrt(sum(x * x for x in a))

def cosine_similarity(a, b):
    """Pure-Python cosine similarity. Returns 0.0 if either vector is all zeros."""
    denom = magnitude(a) * magnitude(b)
    if denom == 0:
        return 0.0
    return dot(a, b) / denom

# Sanity check on tiny by-hand vectors (the same ones you draw on the whiteboard):
cat     = [1.0, 0.9, 0.1]
kitten  = [0.9, 1.0, 0.0]
invoice = [0.1, 0.0, 1.0]
print("cat ~ kitten :", round(cosine_similarity(cat, kitten), 3), "(close meaning → high)")
print("cat ~ invoice:", round(cosine_similarity(cat, invoice), 3), "(unrelated   → low)")

cat ~ kitten : 0.992 (close meaning → high)
cat ~ invoice: 0.148 (unrelated   → low)


### Now on real text

The toy vectors above were made up so you could see the math. Now feed *actual sentences*
through `embed()` and score them. The password question finds the password sentence.

**But read the two texts carefully** — they *share the word "password."* Offline,
`fallback_embed()` is a word-counter, so shared words are exactly how it scored this match.
Nothing deeper happened. So has this demo actually shown "search that understands meaning"?
Not yet. 

In [8]:
query = "how do I change my password?"
library = [
    "Click 'Forgot password' to reset your password.",
    "Full-time employees get 15 vacation days per year.",
    "The Team plan costs 8 dollars per user per month.",
]

q_vec = embed(query)
for chunk in library:
    score = cosine_similarity(q_vec, embed(chunk))
    print(f"{score:5.3f}  |  {chunk}")

0.535  |  Click 'Forgot password' to reset your password.
0.000  |  Full-time employees get 15 vacation days per year.
0.000  |  The Team plan costs 8 dollars per user per month.


### The honest test: meaning vs. words

What happens when the query shares **zero** words with the right
answer? A colleague asks *"I got locked out of my account"* — a human instantly knows the
password-reset sentence is the answer. It shares **no words** with the query
(`locked`, `account` vs. `forgot`, `password`, `reset`).

Run the test:

In [9]:
paraphrase = "I got locked out of my account"

q_vec = embed(paraphrase)
for chunk in library:
    score = cosine_similarity(q_vec, embed(chunk))
    print(f"{score:5.3f}  |  {chunk}")

# Which embedder is running right now?
offline = embed is fallback_embed
print()
if offline:
    print("You are OFFLINE (fallback_embed). Look at those scores: the word-counter is lost —")
    print("the RIGHT chunk scores ~0.000 because it shares no words with the query. (A small")
    print("nonzero score on a WRONG chunk is just hash-collision noise — two different words")
    print("landing in the same bucket. The word-counter isn't merely blind here; it can even")
    print("be misled.)")
    print()
    print("THIS GAP is exactly what a real embedding model buys you. Trained on billions of")
    print("sentences, it has learned that 'locked out of my account' and 'reset your password'")
    print("belong together — so their vectors point the same way even with zero shared words.")
    print("At the in-person KEY DAY segment you re-run exactly this cell with the instructor key")
    print("and watch the right chunk win.")
else:
    print("You are using the REAL embedding API — and the password chunk should score highest")
    print("despite sharing zero words with the query. THAT is meaning-matching: the model")
    print("learned from billions of sentences that these two ideas belong together.")
    print("(Try it offline too — the fallback word-counter fails this exact test.)")

0.000  |  Click 'Forgot password' to reset your password.
0.000  |  Full-time employees get 15 vacation days per year.
0.174  |  The Team plan costs 8 dollars per user per month.

You are OFFLINE (fallback_embed). Look at those scores: the word-counter is lost —
the RIGHT chunk scores ~0.000 because it shares no words with the query. (A small
nonzero score on a WRONG chunk is just hash-collision noise — two different words
landing in the same bucket. The word-counter isn't merely blind here; it can even
be misled.)

THIS GAP is exactly what a real embedding model buys you. Trained on billions of
sentences, it has learned that 'locked out of my account' and 'reset your password'
belong together — so their vectors point the same way even with zero shared words.
At the in-person KEY DAY segment you re-run exactly this cell with the instructor key
and watch the right chunk win.


**The takeaway:** the offline fallback matches **words**; a real embedding model
matches **meaning**. When search
"feels smart," what you're feeling is a model that passes this paraphrase test.

This is also a *professional* judgment skill: when your assistant retrieves a wrong chunk
(Week 6+), your first diagnostic question is now sharper — *did it fail on meaning, or am I
running the word-counter?*


> **Stretch:** invent a paraphrase test from **your own domain** (a question your colleagues
> would ask, and the answer-sentence that shares no words with it). You'll reuse it in the
> Week 5 assignment and again when you index real documents in Week 6.

### Privacy checkpoint: RAG has two model boundaries

A RAG assistant can call **two different kinds of models**:

1. The **embedding model** turns document chunks and questions into vectors.
2. The **chat model** reads the retrieved chunks and writes the final answer.

Making only the chat model local is therefore **half-local**: document text may already have crossed a network boundary during embedding. A fully local path needs local chat **and** local embeddings (plus local storage). Week 6 traces the actual document path; Week 10 lets you evaluate both swaps.

In [ ]:
# Predict first: which configurations keep document text on this machine?
configurations = [
    {"chat": "API",   "embed": "API"},
    {"chat": "local", "embed": "API"},
    {"chat": "local", "embed": "local"},
]

for config in configurations:
    fully_local = config["chat"] == "local" and config["embed"] == "local"
    label = "fully local" if fully_local else "some text crosses an API boundary"
    print(f"chat={config['chat']:5s}  embed={config['embed']:5s}  -> {label}")

assert sum(c["chat"] == "local" and c["embed"] == "local" for c in configurations) == 1

## 3. The nearest-chunk loop — the retrieval primitive

Scoring is great, but a retriever needs to *return the best one*. The pattern is the classic
**"track the best so far"** loop:

1. Embed the query **once** (don't re-embed it every iteration).
2. Loop over every chunk; score it.
3. If this score beats the best seen so far, remember this chunk.
4. Return the best chunk at the end.

This is the load-bearing function of the whole course. Weeks 6–7 swap the Python list for
files in a Chroma store — but this loop never changes.

In [ ]:
def nearest_chunk(query, chunks):
    """Return the single chunk most similar to the query."""
    query_vec = embed(query)              # embed the query ONCE
    best_chunk, best_score = "", -1.0
    for chunk in chunks:
        score = cosine_similarity(query_vec, embed(chunk))
        if score > best_score:            # is this the best so far?
            best_chunk, best_score = chunk, score
    return best_chunk

print(nearest_chunk("how do I change my password?", library))
print(nearest_chunk("how much time off do I get?", library))
print(nearest_chunk("what does it cost?", library))

**The conceptual unlock:** semantic search is *string assembly + a `for`-loop that scores
options.* No magic. 

> **Try it:** add a chunk of your own to `library` and ask a question it should match.

### ✏️ Your card — Week 5: beat the baseline

Section 2 planted a suspicion: the password match may have scored well **because the two texts
share the word "password"**, not because anything understood meaning. Time to settle it — and
the only honest way is to compare against a **baseline**: a deliberately dumb method that
counts shared words. "Smarter" means nothing without "smarter than *what*".

The cell below scores a paraphrase query (one that shares **no** words with the chunk that
*should* win) two ways: word overlap, and your embedding retriever.

**Predict first:** which method picks the right chunk? Write it down, then run.

- **Hypothesis:** on a zero-overlap paraphrase, the word-count baseline picks ____ and the
  embedding retriever picks ____
- **Change:** the scoring method (word overlap vs. cosine similarity on embeddings).
- **Held fixed:** the query, the library of chunks, everything else.
- **Metric & prediction:** which chunk each method returns **and the score it won by**. I predict ____
- **Result & decision:** ____

> **Read the scores, not just the picks.** Offline you're running `fallback_embed()`, which is
> itself a word-counter — so on a true paraphrase it typically scores **every chunk 0.000** and
> the "winner" is just whichever chunk came first in the list. It can hand you the *right*
> answer for *no reason*.

In [ ]:
# The dumb ruler: how many query words literally appear in the chunk?
def overlap_score(query, chunk):
    return sum(word in chunk.lower() for word in query.lower().split())

paraphrase = "I got locked out and cannot sign in"   # shares no words with the reset chunk
print(f"query: {paraphrase!r}\n")
q_vec = embed(paraphrase)
scores = [cosine_similarity(q_vec, embed(chunk)) for chunk in library]
for chunk, score in zip(library, scores):
    print(f"overlap={overlap_score(paraphrase, chunk):>2}   cosine={score:5.3f}   |  {chunk}")

print("\nbaseline picks :", max(library, key=lambda c: overlap_score(paraphrase, c)))
print("embeddings pick:", nearest_chunk(paraphrase, library), f"(top cosine {max(scores):.3f})")

# Read the SCORES, not just the pick — this is the trap the card is really about:
if max(scores) == 0.0:
    print("\n!!  Every cosine is 0.000: the embedder found no shared words anywhere, so that")
    print("    'pick' is simply the FIRST chunk in the list winning a tie. It may even be the")
    print("    right answer — for no reason at all. A pick you can't explain is not a result.")


## 4. The NumPy version — the real path once data gets big

Our pure-Python `cosine_similarity` loops over every number in Python. That's perfect for
*understanding*, but with thousands of 1536-dim vectors it gets slow. **NumPy** does the same
arithmetic in fast, vectorized C code:

- `np.dot(a, b)` (or `a @ b`) — the dot product, no Python loop.
- `np.linalg.norm(a)` — the magnitude, no Python loop.

**Same math, identical answer, far faster.** This is where NumPy lives in this course: it's
the production path for the cosine you just built by hand.

> The cell below is **guarded** with `try/except ImportError`. If NumPy isn't installed it
> prints a friendly note and skips — the pure-Python path above still works. Install with
> `uv pip install numpy` to run it.

In [ ]:
def cosine_similarity_numpy(a, b):
    """The same cosine similarity, computed with NumPy."""
    import numpy as np
    va, vb = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0:
        return 0.0
    return float(va @ vb / denom)            # va @ vb is np.dot(va, vb)

try:
    import numpy as np  # noqa: F401
    q_vec = embed("how do I change my password?")
    for chunk in library:
        c_vec = embed(chunk)
        pure = cosine_similarity(q_vec, c_vec)
        fast = cosine_similarity_numpy(q_vec, c_vec)
        print(f"pure={pure:5.3f}  numpy={fast:5.3f}  agree={math.isclose(pure, fast, rel_tol=1e-9)}  |  {chunk}")
except ImportError:
    print("NumPy isn't installed — skipping the NumPy demo.")
    print("The pure-Python cosine above still works. To run this, do: uv pip install numpy")

### Why bother with NumPy?

For 3 chunks you'd never notice. For 10,000 chunks of 1536-dim embeddings, the Python loop
crawls and NumPy flies, because NumPy moves the per-number arithmetic out of Python into
optimized C. NumPy just runs it fast at scale.


> **Studio update:** Live-model exploration is optional; no Key Day table or live call is required. See the Module 5 Studio folder for the current paired activity.

## Recap & what's next

**You built the retrieval engine of RAG:**

- An **embedding** turns text into a vector (`embed` / `fallback_embed`) — and you can now say
  **where those numbers come from**: the Week-2 tokenizer, then **attention** adjusting each
  token's vector by the tokens around it, stacked into a **transformer** (§1a). That missing
  adjustment step is exactly why the hashed word-counter puts *river bank* and *money bank* in
  the same place.
- **`cosine_similarity`** (dot ÷ the two magnitudes) scores how similar two vectors are — you
  built it by hand in pure Python, then saw the **NumPy** version (`np.dot` / `np.linalg.norm`).
- **`nearest_chunk`** is a `for`-loop that scores every chunk and returns the best — the
  retrieval primitive.
- **The honest test:** the offline `fallback_embed()` matches **shared words** and fails a
  zero-overlap paraphrase; a **real embedding model** passes it. That gap — meaning vs. words —
  is precisely what you pay an embedding model for. At the in-person **Key Day** segment you see
  it live: one real `chat()` call next to `FakeLLM`, real embeddings next to the fallback, and
  your own 3-row table of answer quality, latency, and cost.

**The big idea:** *search = string assembly + a `for`-loop that scores options — and "feels
smart" = passes the paraphrase test.*


**Today's words:** *embedding · vector · dimension · cosine similarity · magnitude · attention ·
transformer · retrieval primitive.*

**Next week (Week 6):** load real `.txt`/`.md` files, chunk them, and store the vectors in a
persistent **Chroma** index — the same idea, scaled from a Python list to real documents.